In [2]:
!pip install -q -U peft bitsandbytes datasets accelerate transformers
!pip uninstall -y -q torchao

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
import os, re, json, time, gc, random, math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
N_TOTAL = 2000
N_TEST = 200
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
WARMUP = 0.05
MAX_LEN = 640
MAX_NEW = 128
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
DEVICE = "cuda"

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
print(torch.cuda.get_device_name(0))

Tesla T4


In [4]:
raw = load_dataset("glaiveai/glaive-function-calling-v2", split="train")

def parse_example(row):
    chat = row["chat"]
    system = row["system"].replace("SYSTEM:", "", 1).strip()
    m = re.search(r"USER:\s*(.*?)\s*\n\s*\n\s*\n?\s*ASSISTANT:\s*(.*?)(?:<\|endoftext\|>|$)", chat, re.S)
    if not m:
        return None
    user = m.group(1).strip()
    reply = m.group(2).strip()
    if not reply.startswith("<functioncall>"):
        return None
    body = reply[len("<functioncall>"):].strip()
    name = re.search(r'"name":\s*"([^"]+)"', body)
    args = re.search(r'"arguments":\s*(?:\'(.*)\'|"(.*)")\s*\}\s*$', body, re.S)
    if not name or not args:
        return None
    arg_text = args.group(1) if args.group(1) is not None else args.group(2)
    try:
        arg_obj = json.loads(arg_text)
    except Exception:
        try:
            arg_obj = json.loads(arg_text.replace('\\"', '"'))
        except Exception:
            return None
    if not isinstance(arg_obj, dict):
        return None
    return {"system": system, "user": user, "name": name.group(1), "args": arg_obj}

FORMAT = "\n\nWhen a function is needed, reply with only one JSON object of the form {\"name\": \"<function name>\", \"arguments\": {...}} and nothing else."

def build_prompt(ex):
    msgs = [{"role": "system", "content": ex["system"] + FORMAT}, {"role": "user", "content": ex["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def target_text(ex):
    return json.dumps({"name": ex["name"], "arguments": ex["args"]}, ensure_ascii=False)

def total_len(ex):
    return len(tok(build_prompt(ex) + target_text(ex) + "<|im_end|>", add_special_tokens=False)["input_ids"])

idx = list(range(len(raw)))
random.Random(SEED).shuffle(idx)
data = []
for i in idx:
    ex = parse_example(raw[i])
    if ex is not None and total_len(ex) <= MAX_LEN:
        data.append(ex)
    if len(data) == N_TOTAL:
        break

train_data = data[:N_TOTAL - N_TEST]
test_data = data[N_TOTAL - N_TEST:]
print(len(train_data), len(test_data))
print(test_data[0]["user"])
print(test_data[0]["name"], test_data[0]["args"])

README.md:   0%|          | 0.00/106 [00:00<?, ?B/s]

glaive-function-calling-v2.json: reconstructing file:   0%|          |  0.00B /  271MB            

glaive-function-calling-v2.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112960 [00:00<?, ? examples/s]

1800 200
Hi, I need to convert 1000 USD to Euros. Can you help me with that?
convert_currency {'amount': 1000, 'from_currency': 'USD', 'to_currency': 'EUR'}


In [5]:
class SFTData(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.items = []
        for ex in rows:
            p = tok(build_prompt(ex), add_special_tokens=False)["input_ids"]
            t = tok(target_text(ex) + "<|im_end|>", add_special_tokens=False)["input_ids"]
            ids = (p + t)[:MAX_LEN]
            lab = ([-100] * len(p) + t)[:MAX_LEN]
            self.items.append((ids, lab))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    m = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), m), tok.pad_token_id)
    lab = torch.full((len(batch), m), -100)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    for j, (a, b) in enumerate(batch):
        ids[j, :len(a)] = torch.tensor(a)
        lab[j, :len(b)] = torch.tensor(b)
        att[j, :len(a)] = 1
    return ids, att, lab

In [6]:
def extract_name(text):
    m = re.search(r'"name"\s*:\s*"([^"]+)"', text)
    return m.group(1) if m else None

def evaluate(model, rows, bs=8):
    model.eval()
    outs = []
    for s in range(0, len(rows), bs):
        batch = rows[s:s + bs]
        enc = tok([build_prompt(e) for e in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True, eos_token_id=IM_END, pad_token_id=tok.pad_token_id)
        for g in gen[:, enc["input_ids"].shape[1]:]:
            outs.append(tok.decode(g, skip_special_tokens=True).strip())
    valid = 0
    tool_ok = 0
    full_ok = 0
    records = []
    for e, o in zip(rows, outs):
        parsed = None
        is_valid = False
        try:
            parsed = json.loads(o)
            is_valid = True
        except Exception:
            parsed = None
        if isinstance(parsed, dict):
            pname = parsed.get("name")
            pargs = parsed.get("arguments")
        else:
            pname = extract_name(o)
            pargs = None
        ok_tool = pname == e["name"]
        ok_full = is_valid and isinstance(parsed, dict) and ok_tool and pargs == e["args"]
        valid += int(is_valid)
        tool_ok += int(ok_tool)
        full_ok += int(ok_full)
        records.append({"user": e["user"], "expected_name": e["name"], "expected_args": e["args"], "output": o, "valid_json": is_valid, "tool_ok": ok_tool, "full_ok": ok_full})
    n = len(rows)
    metrics = {"json_valid": valid, "tool_correct": tool_ok, "full_match": full_ok, "json_valid_pct": 100 * valid / n, "tool_acc_pct": 100 * tool_ok / n, "full_match_pct": 100 * full_ok / n}
    return metrics, records

In [7]:
def count_params(model):
    trainable = 0
    total = 0
    for p in model.parameters():
        n = p.numel()
        if p.__class__.__name__ == "Params4bit":
            n = n * 2
        total += n
        if p.requires_grad:
            trainable += n
    return trainable, total

def load_base(quant):
    if quant:
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=cfg, device_map={"": 0})
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=False)
    else:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map={"": 0})
    m.config.use_cache = False
    return m

def make_peft(method, r, total_steps):
    if method == "adalora":
        return AdaLoraConfig(init_r=int(1.5 * r), target_r=r, lora_alpha=2 * r, target_modules=TARGETS, lora_dropout=0.05,
                             tinit=max(1, int(0.1 * total_steps)), tfinal=max(1, int(0.2 * total_steps)),
                             deltaT=max(1, int(0.05 * total_steps)), total_step=total_steps, task_type="CAUSAL_LM")
    return LoraConfig(r=r, lora_alpha=2 * r, target_modules=TARGETS, lora_dropout=0.05, use_dora=(method == "dora"), task_type="CAUSAL_LM")

def run_method(method, r, tag):
    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()
    model = load_base(method == "qlora")
    ds = SFTData(train_data)
    g = torch.Generator()
    g.manual_seed(SEED)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=g)
    total_steps = (len(loader) // GRAD_ACCUM) * EPOCHS
    model = get_peft_model(model, make_peft(method, r, total_steps))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    trainable, total = count_params(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler()
    model.train()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    step = 0
    micro = 0
    for epoch in range(EPOCHS):
        for ids, att, lab in loader:
            ids, att, lab = ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            micro += 1
            if micro % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                step += 1
                if method == "adalora":
                    model.base_model.update_and_allocate(step)
                opt.zero_grad(set_to_none=True)
                if step % 20 == 0:
                    print(tag, "step", step, "of", total_steps, "loss", round(loss.item() * GRAD_ACCUM, 4))
    minutes = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    metrics, records = evaluate(model, test_data)
    row = {"run": tag, "method": method, "rank": r, "trainable": trainable, "total": total,
           "trainable_pct": 100 * trainable / total, "peak_gb": peak, "minutes": minutes, **metrics}
    del model, opt, loader, scaler, sched
    gc.collect()
    torch.cuda.empty_cache()
    return row, records

def run_base():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map={"": 0})
    metrics, records = evaluate(model, test_data)
    total = sum(p.numel() for p in model.parameters())
    row = {"run": "base", "method": "base", "rank": 0, "trainable": 0, "total": total, "trainable_pct": 0.0,
           "peak_gb": float("nan"), "minutes": 0.0, **metrics}
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return row, records

In [8]:
def safe_run(method, r, tag):
    try:
        return run_method(method, r, tag)
    except Exception as e:
        print("FAILED", tag, type(e).__name__, str(e)[:300])
    gc.collect()
    torch.cuda.empty_cache()
    return None, None

lens = [total_len(e) for e in train_data]
longest = [train_data[i] for i in np.argsort(lens)[-64:]]
full_train, full_test = train_data, test_data
train_data, test_data = longest, full_test[:8]
smoke_ok = True
for method, r in [("lora", 8), ("qlora", 8), ("dora", 8), ("adalora", 8), ("lora", 32), ("qlora", 32), ("dora", 32), ("adalora", 32)]:
    row, rec = safe_run(method, r, "smoke_" + method + "_r" + str(r))
    if row is None:
        smoke_ok = False
    else:
        print("smoke ok", method, r, round(row["peak_gb"], 2), "GB", round(row["minutes"], 2), "min")
train_data, test_data = full_train, full_test
print("ALL SMOKE TESTS PASSED" if smoke_ok else "SOME SMOKE TESTS FAILED ")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/tmp/ipykernel_423/2320618709.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


... (11 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

smoke ok dora 32 8.98 GB 0.33 min


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

smoke ok adalora 32 8.52 GB 0.34 min
ALL SMOKE TESTS PASSED


In [ ]:
results = []
all_records = {}

row, rec = run_base()
results.append(row)
all_records[row["run"]] = rec
print(row)

for method, r in [("lora", 8), ("qlora", 8), ("dora", 8), ("adalora", 8)]:
    row, rec = safe_run(method, r, method + "_r" + str(r))
    if row is None:
        continue
    results.append(row)
    all_records[row["run"]] = rec
    print(row)
    pd.DataFrame(results).to_csv("q2_results.csv", index=False)

df = pd.DataFrame(results)
df

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'run': 'base', 'method': 'base', 'rank': 0, 'trainable': 0, 'total': 1543714304, 'trainable_pct': 0.0, 'peak_gb': nan, 'minutes': 0.0, 'json_valid': 200, 'tool_correct': 200, 'full_match': 191, 'json_valid_pct': 100.0, 'tool_acc_pct': 100.0, 'full_match_pct': 95.5}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/tmp/ipykernel_423/2320618709.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


lora_r8 step 20 of 112 loss 0.0027
lora_r8 step 40 of 112 loss 0.0001
lora_r8 step 60 of 112 loss 0.0002


In [11]:
tuned = df[df["method"] != "base"].sort_values(["tool_acc_pct", "full_match_pct"], ascending=False)
best_method = tuned.iloc[0]["method"]
print("best method:", best_method)

for r in [4, 16, 32]:
    row, rec = safe_run(best_method, r, best_method + "_r" + str(r))
    if row is None:
        continue
    results.append(row)
    all_records[row["run"]] = rec
    print(row)
    pd.DataFrame(results).to_csv("q2_results.csv", index=False)

df = pd.DataFrame(results)
df

best method: lora


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/tmp/ipykernel_423/2320618709.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


lora_r4 step 20 of 112 loss 0.0037
lora_r4 step 40 of 112 loss 0.0001
lora_r4 step 60 of 112 loss 0.0002
lora_r4 step 80 of 112 loss 0.0041
lora_r4 step 100 of 112 loss 0.0001
{'run': 'lora_r4', 'method': 'lora', 'rank': 4, 'trainable': 1089536, 'total': 1544803840, 'trainable_pct': 0.07052908413277896, 'peak_gb': 8.136594295501709, 'minutes': 4.915307140350341, 'json_valid': 200, 'tool_correct': 200, 'full_match': 193, 'json_valid_pct': 100.0, 'tool_acc_pct': 100.0, 'full_match_pct': 96.5}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

lora_r16 step 20 of 112 loss 0.0023
lora_r16 step 40 of 112 loss 0.0001
lora_r16 step 60 of 112 loss 0.0002
lora_r16 step 80 of 112 loss 0.0069
lora_r16 step 100 of 112 loss 0.0001
{'run': 'lora_r16', 'method': 'lora', 'rank': 16, 'trainable': 4358144, 'total': 1548072448, 'trainable_pct': 0.28152067467064695, 'peak_gb': 8.194329738616943, 'minutes': 4.974452002843221, 'json_valid': 200, 'tool_correct': 200, 'full_match': 193, 'json_valid_pct': 100.0, 'tool_acc_pct': 100.0, 'full_match_pct': 96.5}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

lora_r32 step 20 of 112 loss 0.0041
lora_r32 step 40 of 112 loss 0.0002
lora_r32 step 60 of 112 loss 0.0004
lora_r32 step 80 of 112 loss 0.0064
lora_r32 step 100 of 112 loss 0.0001
{'run': 'lora_r32', 'method': 'lora', 'rank': 32, 'trainable': 8716288, 'total': 1552430592, 'trainable_pct': 0.5614607213305933, 'peak_gb': 8.271363735198975, 'minutes': 4.978899312019348, 'json_valid': 200, 'tool_correct': 200, 'full_match': 193, 'json_valid_pct': 100.0, 'tool_acc_pct': 100.0, 'full_match_pct': 96.5}


,run,method,rank,trainable,total,trainable_pct,peak_gb,minutes,json_valid,tool_correct,full_match,json_valid_pct,tool_acc_pct,full_match_pct
0,base,base,0,0,1543714304,0.000000,NaN,0.000000,200,200,191,100.0,100.0,95.5
1,lora_r8,lora,8,2179072,1545893376,0.140959,8.155839,5.202966,200,200,192,100.0,100.0,96.0
2,qlora_r8,qlora,8,2179072,1545893376,0.140959,8.949591,7.180383,200,200,192,100.0,100.0,96.0
3,dora_r8,dora,8,2279424,1545993728,0.147441,8.905955,6.560657,200,200,192,100.0,100.0,96.0
4,adalora_r8,adalora,8,3270064,1546984368,0.211383,8.414603,6.440175,200,200,191,100.0,100.0,95.5
5,lora_r4,lora,4,1089536,1544803840,0.070529,8.136594,4.915307,200,200,193,100.0,100.0,96.5
6,lora_r16,lora,16,4358144,1548072448,0.281521,8.194330,4.974452,200,200,193,100.0,100.0,96.5
7,lora_r32,lora,32,8716288,1552430592,0.561461,8.271364,4.978899,200,200,193,100.0,100.0,96.5


In [ ]:
import matplotlib.pyplot as plt

sweep = df[df["method"] == best_method].sort_values("rank")
plt.figure(figsize=(6, 4))
plt.plot(sweep["rank"], sweep["tool_acc_pct"], marker="o")
plt.xscale("log", base=2)
plt.xticks(sweep["rank"], sweep["rank"].astype(int))
plt.xlabel("rank r")
plt.ylabel("tool accuracy (%)")
plt.title("Tool accuracy vs rank (" + best_method + ")")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("q2_rank_vs_accuracy.png", dpi=200)
plt.show()
sweep[["rank", "trainable", "peak_gb", "minutes", "json_valid_pct", "tool_acc_pct", "full_match_pct"]]

In [13]:
best_run = best_method + "_r8"
wrong = [x for x in all_records[best_run] if not x["full_ok"]]
print(len(wrong), "wrong out of", len(all_records[best_run]))
for x in wrong[:5]:
    print("customer:", x["user"])
    print("expected:", x["expected_name"], x["expected_args"])
    print("model   :", x["output"])
    print()
json.dump(wrong, open("q2_errors.json", "w"), indent=2)
json.dump(all_records, open("q2_all_predictions.json", "w"))
df.to_csv("q2_results.csv", index=False)

8 wrong out of 200
customer: I need to create an invoice for a customer named John Doe. The amount due is $500 and the due date is 30th April.
expected: create_invoice {'customer_name': 'John Doe', 'amount_due': 500, 'due_date': '30th April'}
model   : {"name": "create_invoice", "arguments": {"customer_name": "John Doe", "amount_due": 500, "due_date": "2022-04-30"}}

customer: I need to book a flight to Paris on 15th of next month.
expected: book_flight {'destination': 'Paris', 'date': '15th of next month'}
model   : {"name": "book_flight", "arguments": {"destination": "Paris", "date": "next month"}}

customer: I am looking for a laptop under $1000. Can you help me find one?
expected: search_products {'keywords': 'laptop', 'category': 'Electronics', 'price_range': {'min_price': 0, 'max_price': 1000}}
model   : {"name": "search_products", "arguments": {"keywords": "", "category": "laptop", "price_range": {"min_price": 0, "max_price": 1000}}}

customer: I am looking for a book on quantum